# Requests Toolkit

我们可以使用 Requests [工具包](/docs/concepts/tools/#toolkits) 来构建生成 HTTP 请求的代理。

有关所有 API 工具包功能和配置的详细文档，请参阅 [RequestsToolkit](https://python.langchain.com/api_reference/community/agent_toolkits/langchain_community.agent_toolkits.openapi.toolkit.RequestsToolkit.html) 的 API 参考。

## ⚠️ 安全提示 ⚠️
让模型自行决定执行现实世界操作存在固有的风险。请采取预防措施来减轻这些风险：

- 确保与工具关联的权限范围狭窄（例如，针对数据库操作或 API 请求）；
- 在需要时，利用人工审核工作流。

## 设置

### 安装

此工具包位于 `langchain-community` 包中：

In [ ]:
%pip install -qU langchain-community

要启用对单个工具的自动化跟踪，请设置您的 [LangSmith](https://docs.smith.langchain.com/) API 密钥：

In [ ]:
# os.environ["LANGSMITH_API_KEY"] = getpass.getpass("Enter your LangSmith API key: ")
# os.environ["LANGSMITH_TRACING"] = "true"

## 实例化

首先，我们将演示一个最小化的示例。

**注意**: 让模型拥有执行现实世界操作的自由裁量权存在固有风险。我们必须通过设置 `allow_dangerous_request=True` 来“选择加入”这些风险，以便使用这些工具。
**这可能导致调用不受欢迎的请求，存在危险**。请确保您的自定义 OpenAPI 规范 (yaml) 是安全的，并且与工具关联的权限是经过严格限定的。

In [1]:
ALLOW_DANGEROUS_REQUEST = True

我们可以使用 [JSONPlaceholder](https://jsonplaceholder.typicode.com) API 作为测试平台。

让我们创建（其 API 规范的）子集：

In [2]:
from typing import Any, Dict, Union

import requests
import yaml


def _get_schema(response_json: Union[dict, list]) -> dict:
    if isinstance(response_json, list):
        response_json = response_json[0] if response_json else {}
    return {key: type(value).__name__ for key, value in response_json.items()}


def _get_api_spec() -> str:
    base_url = "https://jsonplaceholder.typicode.com"
    endpoints = [
        "/posts",
        "/comments",
    ]
    common_query_parameters = [
        {
            "name": "_limit",
            "in": "query",
            "required": False,
            "schema": {"type": "integer", "example": 2},
            "description": "Limit the number of results",
        }
    ]
    openapi_spec: Dict[str, Any] = {
        "openapi": "3.0.0",
        "info": {"title": "JSONPlaceholder API", "version": "1.0.0"},
        "servers": [{"url": base_url}],
        "paths": {},
    }
    # Iterate over the endpoints to construct the paths
    for endpoint in endpoints:
        response = requests.get(base_url + endpoint)
        if response.status_code == 200:
            schema = _get_schema(response.json())
            openapi_spec["paths"][endpoint] = {
                "get": {
                    "summary": f"Get {endpoint[1:]}",
                    "parameters": common_query_parameters,
                    "responses": {
                        "200": {
                            "description": "Successful response",
                            "content": {
                                "application/json": {
                                    "schema": {"type": "object", "properties": schema}
                                }
                            },
                        }
                    },
                }
            }
    return yaml.dump(openapi_spec, sort_keys=False)


api_spec = _get_api_spec()

接下来我们可以实例化工具包。此 API 不需要任何授权或其他标头：

In [3]:
from langchain_community.agent_toolkits.openapi.toolkit import RequestsToolkit
from langchain_community.utilities.requests import TextRequestsWrapper

toolkit = RequestsToolkit(
    requests_wrapper=TextRequestsWrapper(headers={}),
    allow_dangerous_requests=ALLOW_DANGEROUS_REQUEST,
)

## 工具

查看可用工具：

In [4]:
tools = toolkit.get_tools()

tools

[RequestsGetTool(requests_wrapper=TextRequestsWrapper(headers={}, aiosession=None, auth=None, response_content_type='text', verify=True), allow_dangerous_requests=True),
 RequestsPostTool(requests_wrapper=TextRequestsWrapper(headers={}, aiosession=None, auth=None, response_content_type='text', verify=True), allow_dangerous_requests=True),
 RequestsPatchTool(requests_wrapper=TextRequestsWrapper(headers={}, aiosession=None, auth=None, response_content_type='text', verify=True), allow_dangerous_requests=True),
 RequestsPutTool(requests_wrapper=TextRequestsWrapper(headers={}, aiosession=None, auth=None, response_content_type='text', verify=True), allow_dangerous_requests=True),
 RequestsDeleteTool(requests_wrapper=TextRequestsWrapper(headers={}, aiosession=None, auth=None, response_content_type='text', verify=True), allow_dangerous_requests=True)]

- [RequestsGetTool](https://python.langchain.com/api_reference/community/tools/langchain_community.tools.requests.tool.RequestsGetTool.html)
- [RequestsPostTool](https://python.langchain.com/api_reference/community/tools/langchain_community.tools.requests.tool.RequestsPostTool.html)
- [RequestsPatchTool](https://python.langchain.com/api_reference/community/tools/langchain_community.tools.requests.tool.RequestsPatchTool.html)
- [RequestsPutTool](https://python.langchain.com/api_reference/community/tools/langchain_community.tools.requests.tool.RequestsPutTool.html)
- [RequestsDeleteTool](https://python.langchain.com/api_reference/community/tools/langchain_community.tools.requests.tool.RequestsDeleteTool.html)

## 在代理中使用

You can use the `AgentExecutor` to run an agent.

```python
from langchain.agents import AgentExecutor, create_react_agent
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

# Define the tools
tools = [...] # Your list of tools

# Create the agent
agent = create_react_agent(
    llm=ChatOpenAI(model="gpt-3.5-turbo"),
    tools=tools,
    prompt=PromptTemplate(
        template="""
Your task is to help users with their queries.

Available tools: {tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the output to the user

Begin!

Question: {input}
{agent_scratchpad}
""",
        input_variables=["input", "agent_scratchpad", "tools", "tool_names"],
    ),
)

# Create the AgentExecutor
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True, # Set to True to see the agent's thought process
)

# Run the agent
agent_executor.invoke({"input": "Who is the president of the United States?"})
```

This will run the agent and print the thought process to the console if `verbose` is set to `True`.

### AgentExecutor

The `AgentExecutor` is the class that runs the agent. It takes the agent and a list of tools as input.

- **`agent`**: The agent to run. This is typically created using one of the `create_react_agent` or `create_structured_chat_agent` functions.
- **`tools`**: A list of tools that the agent can use.
- **`verbose`**: If `True`, the agent will print its thought process to the console.
- **`handle_parsing_errors`**: If `True`, the agent will attempt to recover from parsing errors.
- **`max_iterations`**: The maximum number of iterations the agent can run.
- **`max_execution_time`**: The maximum execution time for the agent.
- **`early_stopping_method`**: The method to use for early stopping. Can be "force" or "generate".
- **`agent_kwargs`**: Keyword arguments to pass to the agent.

### Streaming

You can also stream the output of the agent.

```python
for chunk in agent_executor.stream({"input": "Who is the president of the United States?"}):
    print(chunk)
```

This will print each chunk of the agent's output as it is generated.

### Async

You can also run the agent asynchronously.

```python
await agent_executor.ainvoke({"input": "Who is the president of the United States?"})
```

This will run the agent asynchronously and return the result.

### Customizing the Agent

You can customize the agent by providing your own prompt template.

```python
prompt = PromptTemplate(
    template="""
You are a helpful assistant.

Available tools: {tools}

Use the following format:

Question: the input question you must answer
Thought: you should always think about what to do
Action: the action to take, should be one of [{tool_names}]
Action Input: the input to the action
Observation: the result of the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I now know the final answer
Final Answer: the output to the user

Begin!

Question: {input}
{agent_scratchpad}
""",
    input_variables=["input", "agent_scratchpad", "tools", "tool_names"],
)

agent = create_react_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools)
```

### Error Handling

The `AgentExecutor` has built-in error handling. If an error occurs, the agent will attempt to recover from it.

You can also provide your own error handling logic by implementing a custom `AgentExecutor`.

```python
class CustomAgentExecutor(AgentExecutor):
    def _handle_error(self, error: Exception, **kwargs) -> Dict[str, str]:
        # Custom error handling logic here
        return {"output": "An error occurred."}

custom_agent_executor = CustomAgentExecutor(agent=agent, tools=tools)
```

### Agent Scratchpad

The agent scratchpad is a variable that is used to store the agent's thought process. It is a dictionary that contains the following keys:

- **`input`**: The input to the agent.
- **`intermediate_steps`**: A list of tuples, where each tuple contains the action and the observation.

The agent scratchpad is used to provide context to the agent and to allow it to learn from its mistakes.

### Tools

Tools are functions that the agent can use to interact with the outside world. They can be used to perform a variety of tasks, such as:

- Searching the web
- Calling an API
- Accessing a database

Tools are defined as a list of `Tool` objects. Each `Tool` object has the following attributes:

- **`name`**: The name of the tool.
- **`func`**: The function to call.
- **`description`**: A description of the tool.

For example, here is a simple tool that returns the current date:

```python
from langchain.tools import Tool

def get_current_date():
    return datetime.now().strftime("%Y-%m-%d")

tool = Tool(
    name="get_current_date",
    func=get_current_date,
    description="Returns the current date.",
)
```

You can then pass a list of these tools to the `AgentExecutor`.

```python
tools = [tool]
agent_executor = AgentExecutor(agent=agent, tools=tools)
```

### Agent Types

There are several types of agents that you can use:

- **ReAct**: This agent uses the ReAct framework to reason about the problem and decide which tool to use.
- **Structured Chat**: This agent uses a structured chat format to communicate with the user.

You can create these agents using the `create_react_agent` and `create_structured_chat_agent` functions, respectively.

### Prompt Engineering

Prompt engineering is the process of designing effective prompts for language models. It is an important part of using agents, as the prompt determines how the agent will behave.

Here are some tips for prompt engineering:

- **Be clear and concise**: The prompt should be easy to understand and should not contain any ambiguity.
- **Provide context**: The prompt should provide enough context for the agent to understand the problem.
- **Specify the desired output format**: The prompt should specify the desired output format, such as JSON or a list.
- **Use examples**: The prompt can include examples of how the agent should behave.

### Best Practices

Here are some best practices for using agents:

- **Use verbose mode**: This will help you to understand how the agent is behaving and to debug any issues.
- **Test your tools**: Make sure that your tools are working correctly before using them with an agent.
- **Iterate on your prompts**: Prompt engineering is an iterative process. You will need to experiment with different prompts to find the ones that work best.
- **Consider the user experience**: Make sure that the agent is providing a good user experience.

---

## 在代理中使用

您可以使用 `AgentExecutor` 来运行代理。

```python
from langchain.agents import AgentExecutor, create_react_agent
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

# 定义工具
tools = [...] # 您的工具列表

# 创建代理
agent = create_react_agent(
    llm=ChatOpenAI(model="gpt-3.5-turbo"),
    tools=tools,
    prompt=PromptTemplate(
        template="""
您的任务是帮助用户解决他们的查询。

可用工具：{tools}

请使用以下格式：

问题：您必须回答的输入问题
思考：您应该始终考虑要做什么
操作：要执行的操作，应为以下之一：[{tool_names}]
操作输入：要传递给操作的输入
观察：操作的结果
... （此思考/操作/操作输入/观察可以重复 N 次）
思考：我现在知道了最终答案
最终答案：输出给用户的答案

开始！

问题：{input}
{agent_scratchpad}
""",
        input_variables=["input", "agent_scratchpad", "tools", "tool_names"],
    ),
)

# 创建 AgentExecutor
agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True, # 设置为 True 以查看代理的思考过程
)

# 运行代理
agent_executor.invoke({"input": "Who is the president of the United States?"})
```

如果将 `verbose` 设置为 `True`，这将运行代理并将思考过程打印到控制台。

### AgentExecutor

`AgentExecutor` 是运行代理的类。它接收代理和工具列表作为输入。

- **`agent`**: 要运行的代理。这通常使用 `create_react_agent` 或 `create_structured_chat_agent` 函数之一创建。
- **`tools`**: 代理可以使用的工具列表。
- **`verbose`**: 如果为 `True`，代理将在控制台打印其思考过程。
- **`handle_parsing_errors`**: 如果为 `True`，代理将尝试从解析错误中恢复。
- **`max_iterations`**: 代理可以运行的最大迭代次数。
- **`max_execution_time`**: 代理的最大执行时间。
- **`early_stopping_method`**: 用于提前停止的方法。可以是 "force" 或 "generate"。
- **`agent_kwargs`**: 要传递给代理的关键字参数。

### 流式输出

您也可以流式输出代理的结果。

```python
for chunk in agent_executor.stream({"input": "Who is the president of the United States?"}):
    print(chunk)
```

这将打印代理生成的每个输出块。

### 异步

您也可以异步运行代理。

```python
await agent_executor.ainvoke({"input": "Who is the president of the United States?"})
```

这将异步运行代理并返回结果。

### 自定义代理

您可以通过提供自己的提示模板来定制代理。

```python
prompt = PromptTemplate(
    template="""
您是一个有用的助手。

可用工具：{tools}

请使用以下格式：

问题：您必须回答的输入问题
思考：您应该始终考虑要做什么
操作：要执行的操作，应为以下之一：[{tool_names}]
操作输入：要传递给操作的输入
观察：操作的结果
... （此思考/操作/操作输入/观察可以重复 N 次）
思考：我现在知道了最终答案
最终答案：输出给用户的答案

开始！

问题：{input}
{agent_scratchpad}
""",
    input_variables=["input", "agent_scratchpad", "tools", "tool_names"],
)

agent = create_react_agent(llm, tools, prompt)
agent_executor = AgentExecutor(agent=agent, tools=tools)
```

### 错误处理

`AgentExecutor` 具有内置的错误处理机制。如果发生错误，代理将尝试从中恢复。

您也可以通过实现自定义的 `AgentExecutor` 来提供自己的错误处理逻辑。

```python
class CustomAgentExecutor(AgentExecutor):
    def _handle_error(self, error: Exception, **kwargs) -> Dict[str, str]:
        # 在此处添加自定义错误处理逻辑
        return {"output": "发生错误。"}

custom_agent_executor = CustomAgentExecutor(agent=agent, tools=tools)
```

### Agent Scratchpad

Agent Scratchpad 是一个用于存储代理思考过程的变量。它是一个字典，包含以下键：

- **`input`**: 输入给代理的内容。
- **`intermediate_steps`**: 一个元组列表，其中每个元组包含操作和观察结果。

Agent Scratchpad 用于为代理提供上下文，并使其能够从错误中学习。

### 工具

工具是代理可以用来与外部世界交互的函数。它们可用于执行各种任务，例如：

- 搜索网络
- 调用 API
- 访问数据库

工具被定义为 `Tool` 对象的列表。每个 `Tool` 对象具有以下属性：

- **`name`**: 工具的名称。
- **`func`**: 要调用的函数。
- **`description`**: 工具的描述。

例如，这是一个返回当前日期的简单工具：

```python
from langchain.tools import Tool
from datetime import datetime

def get_current_date():
    return datetime.now().strftime("%Y-%m-%d")

tool = Tool(
    name="get_current_date",
    func=get_current_date,
    description="返回当前日期。",
)
```

然后，您可以将这些工具的列表传递给 `AgentExecutor`。

```python
tools = [tool]
agent_executor = AgentExecutor(agent=agent, tools=tools)
```

### 代理类型

有几种类型的代理可供您使用：

- **ReAct**: 此代理使用 ReAct 框架来推理问题并决定使用哪个工具。
- **Structured Chat**: 此代理使用结构化聊天格式与用户进行通信。

您可以使用 `create_react_agent` 和 `create_structured_chat_agent` 函数分别创建这些代理。

### 提示工程

提示工程是为语言模型设计有效提示的过程。它是使用代理的重要组成部分，因为提示决定了代理的行为方式。

以下是一些提示工程的技巧：

- **清晰简洁**: 提示应易于理解，不应包含任何歧义。
- **提供上下文**: 提示应提供足够的上下文供代理理解问题。
- **指定所需的输出格式**: 提示应指定所需的输出格式，例如 JSON 或列表。
- **使用示例**: 提示可以包含代理应如何行为的示例。

### 最佳实践

以下是使用代理的一些最佳实践：

- **使用详细模式**: 这将帮助您了解代理的行为方式并调试任何问题。
- **测试您的工具**: 在将工具与代理一起使用之前，请确保它们正常工作。
- **迭代您的提示**: 提示工程是一个迭代过程。您需要尝试不同的提示来找到最适合的提示。
- **考虑用户体验**: 确保代理提供良好的用户体验。

In [5]:
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

llm = ChatOpenAI(model="gpt-4o-mini")

system_message = """
You have access to an API to help answer user queries.
Here is documentation on the API:
{api_spec}
""".format(api_spec=api_spec)

agent_executor = create_react_agent(llm, tools, prompt=system_message)

In [6]:
example_query = "Fetch the top two posts. What are their titles?"

events = agent_executor.stream(
    {"messages": [("user", example_query)]},
    stream_mode="values",
)
for event in events:
    event["messages"][-1].pretty_print()

================================ Human Message =================================

Fetch the top two posts. What are their titles?
================================== Ai Message ==================================
Tool Calls:
  requests_get (call_RV2SOyzCnV5h2sm4WPgG8fND)
 Call ID: call_RV2SOyzCnV5h2sm4WPgG8fND
  Args:
    url: https://jsonplaceholder.typicode.com/posts?_limit=2
================================= Tool Message =================================
Name: requests_get

[
  {
    "userId": 1,
    "id": 1,
    "title": "sunt aut facere repellat provident occaecati excepturi optio reprehenderit",
    "body": "quia et suscipit\nsuscipit recusandae consequuntur expedita et cum\nreprehenderit molestiae ut ut quas totam\nnostrum rerum est autem sunt rem eveniet architecto"
  },
  {
    "userId": 1,
    "id": 2,
    "title": "qui est esse",
    "body": "est rerum tempore vitae\nsequi sint nihil reprehenderit dolor beatae ea dolores neque\nfugiat blanditiis voluptate porro vel nihil moles

## API 参考

有关所有 API 工具包功能和配置的详细文档，请参阅 [RequestsToolkit](https://python.langchain.com/api_reference/community/agent_toolkits/langchain_community.agent_toolkits.openapi.toolkit.RequestsToolkit.html) 的 API 参考。